# Notebook 12 — Frequency Response

**Companion to Chapter 12**

This notebook studies how a stabilized vertical-motion loop transmits sinusoidal force disturbances into depth motion. It connects Bode plots, damping, bandwidth, delay, and time-domain simulations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

m = 85.0                 # kg
rho = 1025.0             # kg/m^3
g = 9.80665              # m/s^2
z_star = 20.0            # m, positive downward
V_g0 = 8.0e-3            # m^3 at the surface
p_atm = 101325.0          # Pa
p_star = p_atm + rho*g*z_star
V_g_star = V_g0*p_atm/p_star
k_B = -rho**2*g**2*V_g_star/p_star
a = k_B/m

A = np.array([[0.0, -1.0], [a, 0.0]])
B = np.array([[0.0], [1.0/m]])         # positive input force is upward
C = np.array([[1.0, 0.0]])
D = np.zeros((1, 1))

print(f"Local buoyancy slope k_B = {k_B:.4f} N/m")
print(f"Plant coefficient a = {a:.6f} s^-2")

## 1. Stable closed-loop disturbance model

For PD feedback $u=K_pz-K_dv$, an additive force disturbance $d$ gives

$$G_d(s)=\frac{Z(s)}{D(s)}=-\frac{1}{ms^2+K_ds+(ma+K_p)}.$$$

In [ ]:
p_des = np.array([-0.6,-0.9])
K_d = -m*p_des.sum()
K_p = m*(p_des.prod()-a)
den = [m, K_d, m*a+K_p]
Gd = signal.TransferFunction([-1.0], den)
wn = np.sqrt(a+K_p/m)
zeta = K_d/(2*m*wn)
print(f"wn={wn:.3f} rad/s, zeta={zeta:.3f}")
print("poles:", np.roots(den))
assert np.all(np.roots(den).real < 0)

## 2. Bode magnitude and phase

In [ ]:
w = np.logspace(-2, 2, 800)
w, H = signal.freqresp(Gd, w)
mag = np.abs(H)
phase = np.unwrap(np.angle(H))*180/np.pi
mag_db = 20*np.log10(mag)

fig, axes = plt.subplots(2,1,sharex=True,figsize=(8,7))
axes[0].semilogx(w,mag_db); axes[0].set_ylabel("Magnitude [dB re 1 m/N]")
axes[1].semilogx(w,phase); axes[1].set(xlabel="Angular frequency [rad/s]",ylabel="Phase [deg]")
plt.show()

## 3. Bandwidth relative to the low-frequency response

The common $-3$ dB bandwidth is measured relative to the low-frequency magnitude, not necessarily relative to unity.

In [ ]:
target = mag[0]/np.sqrt(2)
idx = np.where(mag <= target)[0]
w_bw = w[idx[0]] if len(idx) else np.nan
print(f"Low-frequency gain: {mag[0]:.5f} m/N")
print(f"Approximate -3 dB bandwidth: {w_bw:.3f} rad/s")

## 4. Frequency-domain predictions against time histories

After transients decay, the depth amplitude should approach $|G_d(j\omega)|d_0$.

In [ ]:
def sinusoid_test(omega, amp=5.0, cycles=18):
    duration = cycles*2*np.pi/omega
    t = np.linspace(0,duration,int(max(2000,duration*80)))
    d = amp*np.sin(omega*t)
    _, z, _ = signal.lsim(Gd,U=d,T=t)
    tail = t > duration*0.7
    measured = (z[tail].max()-z[tail].min())/2
    predicted = amp*abs(signal.freqresp(Gd,[omega])[1][0])
    return t,z,predicted,measured

tests = [0.15, wn, 3.0]
fig, axes = plt.subplots(3,1,figsize=(8,8))
for ax, omega in zip(axes,tests):
    tt,zz,pred,meas = sinusoid_test(omega)
    ax.plot(tt,zz); ax.set_ylabel("z [m]")
    ax.set_title(f"ω={omega:.2f} rad/s: predicted {pred:.3f} m, measured {meas:.3f} m")
axes[-1].set_xlabel("Time [s]"); plt.tight_layout(); plt.show()

## 5. Delay contributes phase, not magnitude

A pure delay $e^{-s\tau}$ has unit magnitude and phase $-\omega\tau$. It therefore reduces phase margin even though the magnitude curve is unchanged.

In [ ]:
fig, ax = plt.subplots()
for tau in [0.0,0.4,0.8]:
    ax.semilogx(w,-w*tau*180/np.pi,label=f"τ={tau:.1f} s")
ax.set(xlabel="Angular frequency [rad/s]",ylabel="Additional phase [deg]",title="Phase penalty from delay")
ax.legend(); plt.show()

## 6. A breathing-like periodic disturbance

In [ ]:
f_breathe = 0.25
w_breathe = 2*np.pi*f_breathe
d_amp = 2.0
depth_amp = d_amp*abs(signal.freqresp(Gd,[w_breathe])[1][0])
print(f"At {f_breathe:.2f} Hz, a {d_amp:.1f} N disturbance predicts {depth_amp*100:.2f} cm depth amplitude.")

## Engineering exercises

1. Reduce $K_d$ and identify whether a resonant peak appears.
2. Change $K_p$ while preserving damping ratio; describe the bandwidth–effort tradeoff.
3. Convert the breathing frequency to rad/s by hand and verify the code.
4. Find the frequency at which a 0.8 s delay contributes $-45^\circ$ phase.

## Summary

Frequency response is meaningful here because feedback first stabilized the plant. The Bode magnitude predicts disturbance transmission, phase describes timing, and delay consumes phase without attenuating the signal. Chapter 13 replaces the ideal continuously acting controller with delayed, intermittent human action.